# 04 — The 3-qubit repetition code

Phase 4 demo. We build the bit-flip repetition code from `qec.codes.repetition`, verify that every weight-1 X error is corrected, observe the logical flip from a weight-2 error, and plot Monte-Carlo logical error rate vs physical error rate against the analytic `3 p^2 - 2 p^3`.

Read alongside `notes/06-repetition-code.md`.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qec import statevec as sv
from qec.codes import repetition as rep

ATOL = 1e-12
plt.rcParams["figure.figsize"] = (6, 4)


## 1. Encoder produces alpha|000> + beta|111>

By construction the encoder is `CNOT(0->1); CNOT(0->2)` applied to `(alpha|0> + beta|1>) ⊗ |00>`.


In [ ]:
alpha, beta = 0.6, 0.8  # arbitrary unit-norm pair
psi = rep.encode_bitflip(alpha, beta)
print("non-zero amplitudes:")
for k in range(8):
    if abs(psi[k]) > 1e-12:
        print(f"  |{k:03b}> : {psi[k]:.4f}")
expected = np.zeros(8, dtype=complex)
expected[0b000] = alpha
expected[0b111] = beta
assert np.allclose(psi, expected, atol=ATOL)
print("OK")


## 2. Single X errors are detected and corrected

Round-trip: encode, apply X to qubit q, measure the two ZZ syndromes, look up the recovery, apply it. The final state should equal the noiseless encoded state for any q.


In [ ]:
rng = np.random.default_rng(0)
a = rng.normal() + 1j * rng.normal()
b = rng.normal() + 1j * rng.normal()
norm = np.sqrt(abs(a)**2 + abs(b)**2)
a, b = a / norm, b / norm
psi_clean = rep.encode_bitflip(a, b)

print(f"{'X on qubit':>12} | {'syndrome':>10} | recovered?")
print("-" * 44)
for q in range(3):
    pat = [0, 0, 0]; pat[q] = 1
    psi_out, syn = rep.round_trip_with_x_error(a, b, tuple(pat))
    ok = np.allclose(psi_out, psi_clean, atol=ATOL)
    print(f"{q:>12} | {syn!s:>10} | {ok}")
    assert ok


## 3. Weight-2 errors cause logical flips

`X_0 X_1` produces the same syndrome as `X_2` alone. The decoder applies `X_2`, leaving `X_0 X_1 X_2 = logical X`.


In [ ]:
psi_clean_zero = rep.encode_bitflip(1, 0)
psi_clean_one  = rep.encode_bitflip(0, 1)

# Start from |0_L>; inject X_0 X_1; recover.
psi_out, syn = rep.round_trip_with_x_error(1, 0, (1, 1, 0))
print(f"X_0 X_1 syndrome: {syn}")
print(f"final state == |1_L>?  {np.allclose(psi_out, psi_clean_one, atol=ATOL)}")
print(f"final state == |0_L>?  {np.allclose(psi_out, psi_clean_zero, atol=ATOL)}")
assert np.allclose(psi_out, psi_clean_one, atol=ATOL)


## 4. Logical error rate vs physical: Monte-Carlo vs analytic

For independent X errors at rate `p`, the analytic logical error rate is `P_L(p) = 3 p^2 - 2 p^3`. The crossover (`P_L < p`) is at `p < 0.5` — the code helps below the trivial pseudo-threshold of 1/2.


In [ ]:
rng = np.random.default_rng(2026)
ps = np.linspace(0, 0.5, 26)
shots = 50_000
mc = np.array([rep.monte_carlo_logical_error(p, shots, rng=rng) for p in ps])
analytic = np.array([rep.analytic_logical_error_rate(p) for p in ps])

plt.plot(ps, ps, "k:", label="no encoding (y = p)")
plt.plot(ps, analytic, "C0-", label="analytic 3p^2 - 2p^3")
plt.plot(ps, mc, "C1.", label=f"Monte Carlo ({shots} shots)")
plt.xlabel("physical error rate p")
plt.ylabel("logical error rate P_L(p)")
plt.title("3-qubit bit-flip repetition code")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Quantitative gate
err = np.max(np.abs(mc - analytic))
sigma = np.sqrt(0.5 * 0.5 / shots)  # worst-case 1-sigma at p=0.5
print(f"max |MC - analytic| = {err:.4f}; worst-case 1-sigma ~ {sigma:.4f}")
assert err < 5 * sigma
print("OK")


## 5. Phase-flip code is the bit-flip code in the X basis

`encode_phaseflip(a, b)` should equal `H ⊗ H ⊗ H` applied to `encode_bitflip(a, b)`.


In [ ]:
a, b = 0.6, 0.8
psi_phase = rep.encode_phaseflip(a, b)

psi_bit = rep.encode_bitflip(a, b)
for q in range(3):
    psi_bit = sv.apply_1q(psi_bit, sv.H, q)

print("max diff:", np.max(np.abs(psi_phase - psi_bit)))
assert np.allclose(psi_phase, psi_bit, atol=ATOL)

# Logical |0> for the phase-flip code is |+++>
psi_zero_logical = rep.encode_phaseflip(1, 0)
plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
expected = np.kron(plus, np.kron(plus, plus))
assert np.allclose(psi_zero_logical, expected, atol=ATOL)
print("phase-flip |0_L> = |+++>  OK")


## What this notebook gates

- Encoder gives `alpha|000> + beta|111>`.
- Every weight-1 X error is detected and corrected exactly.
- A weight-2 error decodes to the wrong logical state (logical flip), as expected.
- Monte-Carlo logical-error rate matches `3 p^2 - 2 p^3` to within Monte-Carlo error.
- Phase-flip code is bit-flip with H on every qubit.

Next: Phase 5 — Shor [[9,1,3]] (concatenation of bit-flip + phase-flip) and Steane [[7,1,3]] (CSS code from the [7,4] Hamming code).
